# The librarian's log: records, loops and functions

Week 0, Wednesday. The college librarian, before the term review:

> "Every book that goes out is logged. Before the term review I need how many issues we had, which
> department borrows most, and how many days late books came back. And I need the same numbers every
> week."

By the end of this notebook each of those four asks has an answer the library can rerun on next
week's log, built from five ideas: a value and its type, a record, one loop with one running total,
a function that returns, and a traceback read from the bottom up.

In [1]:
import pathlib
import sys

here = pathlib.Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

kit.side_by_side(
    kit.ladder(["the librarian's log", "hands-on: next week's log"], lit=0,
               title="Wednesday's notebooks", show=False),
    kit.flow(["a value", "a record", "one loop", "a function", "a traceback"], title="What this notebook adds, in order", show=False),
)

The setup cell finds the programme's helper, `scripts/c2kit.py`, by walking up from this folder, and loads the log from `../data/`. The log holds eight issues, one dictionary per book lent out.

In [2]:
log = kit.load_records("C2_W00_D03_library_log_STUDENT.py")
print(f"{len(log)} issues loaded")

8 issues loaded


## A value and its type

Every value carries a type, and the type decides what an operator does to it. The log holds whole
numbers in `issue_id` and `days_kept` and text in the other three fields, which matters the moment
anything is added, multiplied or divided.

In [3]:
kit.flow(["a value", "a record", "one loop", "a function", "a traceback"], lit=0, title="Where this section sits")

In [4]:
print("5" * 2, 5 * 2, 7 / 2, 7 // 2)

first = log[0]
kit.table(["field", "value", "type"],
          [(field, repr(value), type(value).__name__) for field, value in first.items()],
          caption="The first issue, field by field")

55 10 3.5 3


field,value,type
issue_id,1,int
student,'S01',str
book,'Algebra',str
dept,'Maths',str
days_kept,12,int


In [5]:
kit.check("days_kept is a whole number in every record",
          all(type(issue["days_kept"]) is int for issue in log))
kit.check("dividing two whole numbers with / gives a float", type(7 / 2) is float, f"7 / 2 = {7 / 2}")

`"5" * 2` repeats the text, `7 / 2` always gives a float, and `7 // 2` drops the remainder. The
interview question this answers is the service-company screen opener from Tuesday's row, [SV]
predict the output of this snippet: name each value's type before reading the operator.

## One record, then the log

A record is a dictionary: each key names a field, and the value sits beside it. The log is a list of
those records, so a position picks a record and a key picks a field inside it. Lists count from 0,
which is why the third issue, Calculus borrowed by S01, is `log[2]`.

In [6]:
kit.flow(["a value", "a record", "one loop", "a function", "a traceback"], lit=1, title="Where this section sits")

In [7]:
third = log[2]
print(third)
print(third["book"], third["days_kept"])
print(len(log), "issues, and", len(third), "fields in each")

{'issue_id': 3, 'student': 'S01', 'book': 'Calculus', 'dept': 'Maths', 'days_kept': 20}
Calculus 20
8 issues, and 5 fields in each


In [8]:
kit.check("the log holds 8 issues", len(log) == 8, f"{len(log)} issues")
kit.check("every record carries the same five keys",
          all(set(issue) == set(log[0]) for issue in log))

## One loop, one running total

A running total makes three moves: it starts before the loop, updates once per record, and is read
after the loop finishes. A count is the same three moves with 1 in the update, so the librarian's
first ask and the total days out come from one shape. The busiest department needs a dictionary as
the running total, and that total starts once per department, inside the loop, the first time the
department appears.

In [9]:
kit.flow(["start before the loop", "update once per record", "read after the loop"], lit=1, title="The three moves of a running total")

In [10]:
issues = 0
days = 0
for issue in log:
    issues = issues + 1
    days = days + issue["days_kept"]
print(issues, "issues,", days, "days out in all")

8 issues, 91 days out in all


In [11]:
counts = {}
for issue in log:
    dept = issue["dept"]
    if dept not in counts:
        counts[dept] = 0
    counts[dept] = counts[dept] + 1
print(counts)

{'Maths': 4, 'Arts': 2, 'Science': 2}


In [12]:
kit.check("the department counts add back to the length of the log",
          sum(counts.values()) == len(log), f"{sum(counts.values())} of {len(log)}")
kit.check("Maths borrows most, with 4 issues", counts["Maths"] == max(counts.values()) == 4)

Three of the librarian's answers now stand on the same three ideas: a record, the list that holds
the records, and one loop carrying one total.

In [13]:
kit.stack(["a value, with its type", "a record: keys and values", "the log: a list of records",
           "one loop, one running total"], lit=3, title="What the first three answers stand on")
kit.table(["the librarian asked", "the answer", "how"],
          [("How many issues?", issues, "a count"),
           ("How many days out in all?", days, "a sum"),
           ("Which department borrows most?", "Maths, with 4", "a count per department")],
          caption="What one loop has answered so far")

the librarian asked,the answer,how
How many issues?,8,a count
How many days out in all?,91,a sum
Which department borrows most?,"Maths, with 4",a count per department


## A function that returns

The librarian wants the same numbers every week, so each loop goes inside a function that takes a
log and hands its answer back with `return`. The caller then decides what to do with the answer:
print it, add it to something, or put it in a report. The first version of the late-days helper
below printed its answer instead, and that is the break this section is built on.

In [14]:
kit.flow(["a value", "a record", "one loop", "a function", "a traceback"], lit=3, title="Where this section sits")

In [15]:
def busiest_department(log):
    counts = {}
    for issue in log:
        dept = issue["dept"]
        if dept not in counts:
            counts[dept] = 0
        counts[dept] = counts[dept] + 1
    best = None
    for dept in counts:
        if best is None or counts[dept] > counts[best]:
            best = dept
    return best

print(busiest_department(log))

Maths


### Break it on purpose: a helper that prints

The loan is 14 days, so Calculus, kept for 20, is 6 days late. This helper prints the days past the
loan instead of returning them, and the loop that adds them up stops on the first late book. The
`try` block keeps the notebook running so the error's last line can be read.

In [16]:
import traceback

def late_days(issue):
    print(issue["days_kept"] - 14)

late = 0
try:
    for issue in log:
        if issue["days_kept"] > 14:
            late = late + late_days(issue)
except TypeError:
    error_lines = traceback.format_exc().strip().splitlines()
    print(error_lines[-1])

6
TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'


The helper printed 6, so it ran. What it handed back was `None`, because a function with no
`return` statement returns `None`, and the line `late = late + late_days(issue)` then tried to add
`None` to a number. The fix is one word.

In [17]:
def late_days(issue):
    return issue["days_kept"] - 14

def total_late_days(log):
    late = 0
    for issue in log:
        if issue["days_kept"] > 14:
            late = late + late_days(issue)
    return late

print(total_late_days(log), "days late across the log")

13 days late across the log


In [18]:
kit.check("late_days hands back a number", type(late_days(log[2])) is int,
          f"late_days(log[2]) = {late_days(log[2])}")
kit.check("13 days late across the log", total_late_days(log) == 13, f"{total_late_days(log)} days")
kit.check("busiest_department hands back a department", busiest_department(log) == "Maths")

A weekly report is exactly this: the same functions called on a new log, with nobody editing a loop.
Next week's log goes in, and the answers come back.

## Reading the traceback

Python's own tutorial puts the rule in one line: "The last line of the error message indicates what
happened." Read that line first, then the line of code it points at, then ask what value each name
on that line held when it ran. The full traceback from the broken helper reads the same way from the
bottom up.

In [19]:
kit.stack(["what happened: the last line", "where: the line and its code", "how it got there: the lines above"], lit=0, title="Read a traceback from the bottom up")

In [20]:
for line in error_lines:
    print(line)

Traceback (most recent call last):
  File "/tmp/ipykernel_2617/4232719479.py", line 10, in <module>
    late = late + late_days(issue)
           ~~~~~^~~~~~~~~~~~~~~~~~
TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'


In [21]:
kit.check("the last line names the error type", error_lines[-1].startswith("TypeError"),
          error_lines[-1])
kit.check("the traceback points at the addition",
          any("late + late_days(issue)" in line for line in error_lines))

The four asks, answered by functions that return, so next week's log gets the same treatment.

In [22]:
kit.flow(["a value", "a record", "one loop", "a function", "a traceback"], lit=4, title="The whole path, from a value to a traceback")
kit.table(["the librarian asked", "the answer", "where it comes from"],
          [("How many issues?", len(log), "len(log)"),
           ("Which department borrows most?", busiest_department(log), "busiest_department(log)"),
           ("How many days late?", total_late_days(log), "total_late_days(log)"),
           ("The same numbers every week", "call the functions on the new log", "return")],
          caption="The librarian's four asks")

the librarian asked,the answer,where it comes from
How many issues?,8,len(log)
Which department borrows most?,Maths,busiest_department(log)
How many days late?,13,total_late_days(log)
The same numbers every week,call the functions on the new log,return


In [23]:
kit.check_summary()
print("Next: the hands-on notebook asks the same questions of the following week's log, where every number is different.")

Next: the hands-on notebook asks the same questions of the following week's log, where every number is different.
